In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os # to read the data
import kagglehub
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import StratifiedKFold


import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the dataset
food_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(food_path)

print(f"Dataset shape: {df.shape}")


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title("Deliver Time Distribution")
plt.xlabel("price")
plt.ylabel("Frequency")
plt.show()

In [ ]:
df_clean = df.copy()

In [ ]:
# Task 1: Write your code here:
df_clean = df_clean.drop('Order_ID',axis=1)

In [ ]:
# Task 2: Write your code here:
null_cat_cols = list(df_clean.select_dtypes(include='object').isnull())
null_num_cols = list(df_clean.select_dtypes(include='number').drop("Delivery_Time", axis=1).isnull())
null_num_cols

In [ ]:
# fill missing values in catigorical columns with the mode of that column
for col in null_cat_cols:
  df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# fill missing values in numerical columns with the mean of that column
for col in null_num_cols:
  avg = np.mean(df_clean[col])
  df_clean[col] = df_clean[col].fillna(avg)


In [ ]:
df_clean.isnull().sum()

In [ ]:
# i will drop the rows that do not have a value for the target
df_clean = df_clean.dropna(subset='Delivery_Time')

In [ ]:
df_clean.isnull().sum()

In [ ]:
# Task 3: Write your code here:

df_clean.duplicated().any()

In [ ]:
# i will drop all the duplicated data
df_clean = df_clean.drop_duplicates()
df_clean.duplicated().any()

In [ ]:
# Task 4: Write your code here:
cat_cols = df_clean.select_dtypes(include='object').columns


In [ ]:
for col in cat_cols:
  enc = LabelEncoder()
  df_clean[col] = enc.fit_transform(df_clean[col])
df_clean.info()

In [ ]:
# Task 5: Write your code here:
feature_cols = df_clean.columns.drop("Delivery_Time")
feature_cols

In [ ]:
scale = StandardScaler()
df_clean[feature_cols] = scale.fit_transform(df_clean[feature_cols])


In [ ]:
df_clean.head()

In [ ]:
# Task 6: Write your code here:
# the targets are mainly between 30 and 95
# so to ensure that the model will not get affected i will treat it as imbalanced
plt.hist(df['Delivery_Time'].dropna(), bins=30, edgecolor='black')
plt.title("Deliver Time Distribution")
plt.xlabel("price")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Task 1: Write your code here:
X = df_clean[feature_cols]
y = df_clean['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestRegressor(n_estimators=200)
all_mae = []
predicted_time = []


for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
  X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
  y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]



  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # mae
  mae = mean_absolute_error(y_test, y_pred)
  all_mae.append(mae)

  # for the plot later
  predicted_time.append(y_pred)


print(f"avg MAE: {np.mean(all_mae)}")



In [ ]:
# Task 1: Write your code here:
importance = model.feature_importances_
plt.barh(feature_cols, importance)
plt.title(f"Random Forst Importance")
plt.xlabel("Coefficient  (Importance)")
plt.tight_layout()
plt.show()

In [ ]:
# Task 5: Write your code here:
plt.hist(predicted_time, edgecolor='black')
plt.title("predicted Deliver Time Distribution")
plt.xlabel("Time")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Task Bonus: Write your code here: